# ⚙️ Module 02: Coroutines, Tasks (`create_task`), and Event Loop Internals

In Module 01, we learned *why* async exists (I/O concurrency without threads). 
In this module, we look under the hood:
0. **🌐 Architectural Comparison**: Python Coroutines vs Go Goroutines vs Kotlin Coroutines
1. **What is a Coroutine object?** (Why calling `foo()` does not actually run `foo`)
2. **What is a `Task`?** (`asyncio.create_task`)
3. **Sequential `await` vs Background `create_task`**
4. **The Infamous Garbage Collection Trap** in production (FastAPI/Django background workers)
5. **Task Lifecycle**: States, Cancelling tasks, and `CancelledError`
6. **🎯 Hands-On Challenge**: Build a Resilient Background Task Manager

---
## 0. 🌐 Architectural Deep Dive: Python Coroutines vs Go Goroutines vs Kotlin Coroutines

A frequent question among backend, mobile, and forward-deployed engineers:
> *"How do Python Coroutines differ from Go's Goroutines and Kotlin's Coroutines?"*

Here is the definitive architectural breakdown:

| Feature | 🐍 Python Coroutines (`asyncio`) | 🐹 Go Goroutines (`go func()`) | 🟣 Kotlin Coroutines (`suspend`) |
|---|---|---|---|
| **Threading Model** | **1 OS Thread** (Single-threaded Event Loop) | **M:N Runtime Scheduler** (M goroutines multiplexed over N OS threads) | **Pool Dispatched** (Multiplexed across thread pools like `Dispatchers.IO`) |
| **Multi-Core Parallelism** | ❌ No (requires `multiprocessing`) | ✅ **Yes** (utilizes all CPU cores automatically) | ✅ **Yes** (via multi-threaded dispatchers) |
| **Preemption** | **Cooperative only** (must hit an `await`) | **Preemptive + Cooperative** (runtime preempts long CPU loops via OS signals) | **Cooperative** (yields at `suspend` points or `yield()`) |
| **Stack Model** | **Stackless** (Generator state machine on heap) | **Stackful** (starts at tiny 2KB dynamic stack, grows/shrinks) | **Stackless** (compiled to state machines passing `Continuation`) |
| **Function Coloring** | 🔴 **Yes** (`async def` functions require `await` and an async caller) | 🟢 **No (Colorless)** (any normal function can run with `go fn()`) | 🔴 **Yes** (`suspend fun` must be called from another suspend or scope) |
| **Concurrency Paradigm** | Event Loop & Callbacks / Tasks | **CSP** (Communicating Sequential Processes) with Channels (`chan`) | **Structured Concurrency** (`CoroutineScope`, parent-child hierarchy) |
| **Memory per Unit** | ~1.5 KB per coroutine | ~2 KB per goroutine | ~1 KB per coroutine |

---

### 🔍 Syntax Comparison Side-by-Side:

#### 1. Python (`asyncio`):
```python
# Must be declared async def (colored function)
async def fetch_user(user_id: int):
    await asyncio.sleep(1.0)
    return {"id": user_id}

# Spawn in background:
task = asyncio.create_task(fetch_user(101))
```

#### 2. Go (`goroutines`):
```go
// No special keyword needed for the function itself!
func fetchUser(userID int) {
    time.Sleep(1 * time.Second)
}

// Spawn in background across ANY available CPU core:
go fetchUser(101)
```

#### 3. Kotlin (`coroutines`):
```kotlin
// Marked with suspend keyword
suspend fun fetchUser(userId: Int): User {
    delay(1000) // Non-blocking delay
    return User(userId)
}

// Spawn in background inside a structured scope:
val job = scope.launch(Dispatchers.IO) {
    fetchUser(101)
}
```

> 💡 **Key Takeaway for Python Engineers**:
> - In **Go and Kotlin**, background tasks can run in **true parallel across multiple CPU cores** because their runtimes manage multi-threaded pools.
> - In **Python `asyncio`**, everything runs on **ONE single thread by default**. That's why blocking the Python event loop with CPU math freezes your entire server, whereas in Go or Kotlin's worker pool, other threads keep moving!

---
## 1. What is a Coroutine Really?

In standard Python, calling `def greet(): return "hi"` executes the function immediately.
In async Python, calling `async def greet(): return "hi"` **DOES NOT RUN THE CODE**.

Instead, Python returns a **suspended coroutine object** waiting to be driven by an Event Loop!

In [ ]:
import asyncio


async def fetch_data(item_id: int) -> dict:
    print(f"Executing fetch_data for ID: {item_id}")
    await asyncio.sleep(0.5)
    return {"id": item_id, "status": "SUCCESS"}

# Look what happens when we call it without await:
coro_obj =  fetch_data(101)
print("1. Calling fetch_data(101) directly WITHOUT await:")
print(f"Type: {type(coro_obj)}")
print(f"Object representation: {coro_obj}")
print("\nNotice: The print inside fetch_data was NEVER executed!")
print("-" * 50)

# Now we drive it to completion:
print("2. Now we await the coroutine:")
result = await coro_obj
print(f"Result: {result}")

1. Calling fetch_data(101) directly WITHOUT await:
Type: <class 'coroutine'>
Object representation: <coroutine object fetch_data at 0x10a3ef5e0>

Notice: The print inside fetch_data was NEVER executed!
--------------------------------------------------
2. Now we await the coroutine:
Executing fetch_data for ID: 101
Result: {'id': 101, 'status': 'SUCCESS'}


---
## 2. `await coro()` vs `asyncio.create_task(coro())`

This is the single most common source of confusion for beginners:

- `await my_coro()`: **Pauses the current function** until `my_coro()` finishes.
- `task = asyncio.create_task(my_coro())`: **Schedules `my_coro()` on the Event Loop immediately in the background**, and returns execution to you **instantly** without waiting!

### 🌍 Real-World Use Case (FastAPI / Web APIs):
When a user signs up (`POST /signup`), we must:
1. Save user to database (50ms) -> Must wait before responding with HTTP 201.
2. Send welcome email (1.5s) -> **Should NOT make the user wait on their screen!**
3. Send analytics event to Mixpanel/Datadog (0.8s) -> **Should NOT make the user wait!**

In [ ]:
import asyncio
import time

async def save_user_to_db(email: str):
    print(f"[{time.strftime('%X')}] 💾 Saving user to Database...")
    await asyncio.sleep(0.05) # Fast 50ms DB write
    print(f"[{time.strftime('%X')}] ✅ User saved to Database!")
    return {"user_id": 99, "email": email}

async def send_welcome_email(email: str):
    await asyncio.sleep(1.5) # Slow 1.5s SMTP call
    print(f"[{time.strftime('%X')}] 📧 [Background] Welcome email delivered to {email}!")

async def track_analytics(event: str):
    await asyncio.sleep(0.8) # 800ms analytics call
    print(f"[{time.strftime('%X')}] 📊 [Background] Analytics event '{event}' delivered!")

print("=" * 60)
print("🚀 USER SIGNUP REQUEST STARTED")
print("=" * 60)
start = time.perf_counter()

# 1. Critical path: save user (must await)
user = await save_user_to_db("alice@example.com")

# 2. Non-critical path: fire & forget in background using create_task!
print(f"[{time.strftime('%X')}] ⚡ Scheduling background email & analytics...")
asyncio.create_task(send_welcome_email(user["email"]))
asyncio.create_task(track_analytics("user_registered"))

# 3. Respond immediately to user
elapsed = time.perf_counter() - start
print(f"[{time.strftime('%X')}] 📬 HTTP 201 Created sent to user! Total response time: {elapsed:.2f}s!")
print("=" * 60)
print("(Meanwhile, in the background...)")

# Wait for background tasks to finish so output displays cleanly in this notebook:
# await asyncio.gather(task1, task2)

🚀 USER SIGNUP REQUEST STARTED
[08:13:48] 💾 Saving user to Database...
[08:13:48] ✅ User saved to Database!
[08:13:48] ⚡ Scheduling background email & analytics...
[08:13:48] 📬 HTTP 201 Created sent to user! Total response time: 0.05s!
(Meanwhile, in the background...)


[08:13:49] 📊 [Background] Analytics event 'user_registered' delivered!
[08:13:50] 📧 [Background] Welcome email delivered to alice@example.com!


---
## 3. 🚨 The Infamous Garbage Collection Trap

In production Python, doing this:
```python
# DANGEROUS IN PRODUCTION:
asyncio.create_task(send_email())
```
has a notorious bug documented in official Python docs:  
> *"An unreferenced Task object can be garbage-collected at any time, even in the middle of executing its coroutine!"*

### 🛡️ The Battle-Tested Production Pattern: Strong References
To guarantee background tasks never disappear mid-execution, maintain a background task set:

In [3]:
# Global set of strong references for active background tasks
ACTIVE_BACKGROUND_TASKS = set()

def safe_background_task(coro) -> asyncio.Task:
    task = asyncio.create_task(coro)
    ACTIVE_BACKGROUND_TASKS.add(task)
    print(f"[{time.strftime('%X')}] ➕ Task {task.get_name()} added to tracking set. Active count: {len(ACTIVE_BACKGROUND_TASKS)}")
    
    # Remove task from set automatically when done
    task.add_done_callback(lambda t: (
        ACTIVE_BACKGROUND_TASKS.discard(t),
        print(f"[{time.strftime('%X')}] ➖ Task {t.get_name()} finished. Discarded from set! Active count: {len(ACTIVE_BACKGROUND_TASKS)}")
    ))
    return task

async def slow_upload(chunk_id: int, delay: float):
    await asyncio.sleep(delay)
    print(f"[{time.strftime('%X')}] 📦 Processed chunk {chunk_id}")

# Launch tasks safely
t1 = safe_background_task(slow_upload(1, 0.5))
t2 = safe_background_task(slow_upload(2, 0.8))

print(f"Currently tracking {len(ACTIVE_BACKGROUND_TASKS)} active background tasks...")
await asyncio.gather(t1, t2)
print("✅ All background tasks finished safely!")

[08:05:03] ➕ Task Task-1 added to tracking set. Active count: 1
[08:05:03] ➕ Task Task-2 added to tracking set. Active count: 2
Currently tracking 2 active background tasks...
[08:05:03] 📦 Processed chunk 1
[08:05:03] ➖ Task Task-1 finished. Discarded from set! Active count: 1
[08:05:04] 📦 Processed chunk 2
[08:05:04] ➖ Task Task-2 finished. Discarded from set! Active count: 0
✅ All background tasks finished safely!


---
## 4. Task Lifecycle: States, Inspection & Cancellation

Tasks have lifecycle states:
- `task.done()`: Returns `True` if completed or cancelled.
- `task.result()`: Returns the return value (or raises exception if it failed).
- `task.cancel()`: Requests cancellation by raising `asyncio.CancelledError` inside the coroutine at its next `await` point.

In [6]:
async def long_polling_worker():
    try:
        print(f"[{time.strftime('%X')}] 📡 Long polling stream starting...")
        while True:
            await asyncio.sleep(1.0)
            print(f"[{time.strftime('%X')}] 📡 Polled 10 new records...")
    except asyncio.CancelledError:
        print(f"[{time.strftime('%X')}] 🧹 Caught CancelledError! Cleaning up database connections and sockets...")
        raise # Always re-raise CancelledError to confirm cancellation to the loop!

# Start task
polling_task = asyncio.create_task(long_polling_worker())
print(f"Is task done right after creation? {polling_task.done()}")

# Let it run for a brief moment
await asyncio.sleep(0.5)

# Cancel it!
print("Triggering cancellation now!")
polling_task.cancel()

try:
    await polling_task
except asyncio.CancelledError:
    print(f"Is task cancelled? {polling_task.cancelled()}")
    print(f"Is task done? {polling_task.done()}")

Is task done right after creation? False
[08:17:17] 📡 Long polling stream starting...
Triggering cancellation now!
[08:17:17] 🧹 Caught CancelledError! Cleaning up database connections and sockets...
Is task cancelled? True
Is task done? True


---
## 5. 🎯 Hands-On Challenge: Build a Background Worker Manager

**Scenario (Forward Deployed Engineering / Microservice Reliability)**:
Your service receives data ingestion events from clients. 
Requirements:
1. Create a `TaskManager` class.
2. `manager.spawn(coro)`: Creates a task, keeps a strong reference, and auto-discards when finished.
3. `manager.active_count`: Returns how many tasks are currently running.
4. `manager.shutdown()`: Cancels all remaining active tasks and waits for them to cleanly exit.

In [5]:
class BackgroundTaskManager:
    def __init__(self):
        self._tasks: set[asyncio.Task] = set()
        
    def spawn(self, coro) -> asyncio.Task:
        task = asyncio.create_task(coro)
        self._tasks.add(task)
        task.add_done_callback(self._tasks.discard)
        return task
        
    @property
    def active_count(self) -> int:
        return len(self._tasks)
        
    async def shutdown(self):
        if not self._tasks:
            return
        print(f"[{time.strftime('%X')}] 🛑 Server shutting down! Cancelling remaining {len(self._tasks)} jobs...")
        for task in self._tasks:
            task.cancel()
        # Await all cancelled tasks with return_exceptions=True so error doesn't crash loop
        await asyncio.gather(*self._tasks, return_exceptions=True)
        self._tasks.clear()

# Test our BackgroundTaskManager:
async def ingestion_job(job_id: int, duration: float):
    try:
        await asyncio.sleep(duration)
        print(f"[{time.strftime('%X')}] 📥 Ingested event {job_id}")
    except asyncio.CancelledError:
        print(f"[{time.strftime('%X')}] ⚠️ Job {job_id} cancelled cleanly.")
        raise

mgr = BackgroundTaskManager()
print(f"[{time.strftime('%X')}] Spawning ingestion jobs...")
mgr.spawn(ingestion_job(1, 0.2))
mgr.spawn(ingestion_job(2, 2.0))
mgr.spawn(ingestion_job(3, 3.0))

print(f"Active jobs currently running: {mgr.active_count}")
await asyncio.sleep(0.4) # Job 1 should finish
print(f"Active jobs after quick job finished: {mgr.active_count}")

# Simulate server shutdown
await mgr.shutdown()
print(f"✅ All jobs safely shut down! Active count: {mgr.active_count}")

[08:05:05] Spawning ingestion jobs...
Active jobs currently running: 3
[08:05:05] 📥 Ingested event 1
Active jobs after quick job finished: 2
[08:05:06] 🛑 Server shutting down! Cancelling remaining 2 jobs...
[08:05:06] ⚠️ Job 2 cancelled cleanly.
[08:05:06] ⚠️ Job 3 cancelled cleanly.
✅ All jobs safely shut down! Active count: 0


---
## 🏆 Summary Checklist for Module 02

- [x] Understood that calling `coro_func()` only creates a **suspended coroutine object**.
- [x] Mastered `asyncio.create_task()` to schedule work immediately without blocking the caller.
- [x] Understood and prevented the **Garbage Collection trap** using strong reference sets.
- [x] Learned how to inspect, cancel, and handle `CancelledError` for graceful shutdowns.
- [x] Built a production-grade `BackgroundTaskManager`.

👉 **Next Up: Module 03 — Modern Task Orchestration, `asyncio.TaskGroup` (Python 3.11+), and ExceptionGroups!**